#  Notebook 06: Production Machine Learning Pipeline Architecture
### End-to-End Feature Engineering & Estimator Pipelines with Scikit-Learn & XGBoost

---

##  Objective & Motivation
In traditional data science workflows, preprocessing, feature transformations, and model fitting are often written as ad-hoc notebook scripts. In **production**, this leads to:
1. **Training-Serving Skew:** Features computed differently during inference than during training.
2. **Data Leakage:** Information from test sets or future time windows leaking into feature scaling or target statistics.
3. **Deployment Fragility:** Complex glue code needed in the REST API to reproduce manual transformations.

**This notebook builds a modular, production-grade Pipeline** using custom Scikit-Learn `BaseEstimator` & `TransformerMixin` classes combined into a unified `Pipeline` object. The entire pipeline from raw timestamps to XGBoost predictions can be trained with `.fit()` and served with `.predict()` in a single step.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

print(" All ML & Pipeline dependencies successfully loaded.")

 All ML & Pipeline dependencies successfully loaded.


## 1. Load Raw Source Data
We load the station hourly traffic data without applying any manual preprocessing outside our pipeline.

In [2]:
# Load base datasets
df_board = pd.read_csv('../data/station-hourly.csv', sep=';')
df_exit = pd.read_csv('../data/station-hourly-exits.csv', sep=';')

df_board.rename(columns={'Ridership': 'Boarding_Count'}, inplace=True)
df_exit.rename(columns={'Ridership': 'Exit_Count'}, inplace=True)

# Merge raw feeds
raw_df = pd.merge(df_board, df_exit, on=['Date', 'Hour', 'Station'], how='outer')
raw_df.dropna(subset=['Boarding_Count'], inplace=True)
raw_df.sort_values(by=['Station', 'Date', 'Hour'], inplace=True)
raw_df.reset_index(drop=True, inplace=True)

print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

Raw dataset shape: (92280, 5)


,Date,Hour,Station,Boarding_Count,Exit_Count
0,2025-08-01,0,Attiguppe,0.0,0
1,2025-08-01,1,Attiguppe,0.0,0
2,2025-08-01,2,Attiguppe,0.0,0
3,2025-08-01,3,Attiguppe,0.0,0
4,2025-08-01,4,Attiguppe,6.0,0


---
## 2. Custom Scikit-Learn Transformers

We encapsulate all domain-specific feature engineering inside reusable Scikit-Learn transformers by inheriting from `BaseEstimator` and `TransformerMixin`.

### 2.1 `TemporalFeatureExtractor`
Transforms `Date` and `Hour` into:
- **Cyclical Sine / Cosine encodings** (ensuring Hour 23 and Hour 0 are continuous in vector space).
- **Rush-Hour Flags:** Morning Commute (8–10 AM) & Evening Surge (5–8 PM).
- **Calendar Components:** DayOfWeek, DayOfMonth, WeekOfYear, Is_Weekend.

In [3]:
class TemporalFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Custom Transformer to extract cyclical and calendar time features.
    """
    def __init__(self, date_col='Date', hour_col='Hour'):
        self.date_col = date_col
        self.hour_col = hour_col

    def fit(self, X, y=None):
        # Stateless transformer
        return self

    def transform(self, X):
        X_out = X.copy()
        
        # Parse Datetime
        dt_series = pd.to_datetime(X_out[self.date_col])
        hours = X_out[self.hour_col].astype(int)
        
        # 1. Cyclical Hour Encoding
        X_out['Hour_Sin'] = np.sin(2 * np.pi * hours / 24.0)
        X_out['Hour_Cos'] = np.cos(2 * np.pi * hours / 24.0)
        
        # 2. Calendar attributes
        dow = dt_series.dt.dayofweek
        X_out['DayOfWeek'] = dow
        X_out['Day_Sin'] = np.sin(2 * np.pi * dow / 7.0)
        X_out['Day_Cos'] = np.cos(2 * np.pi * dow / 7.0)
        X_out['DayOfMonth'] = dt_series.dt.day
        X_out['WeekOfYear'] = dt_series.dt.isocalendar().week.astype(int)
        X_out['Is_Weekend'] = dow.isin([5, 6]).astype(int)
        
        # 3. Peak Commute Windows
        is_wknd = X_out['Is_Weekend'] == 1
        X_out['Is_Morning_Peak'] = ((hours >= 8) & (hours <= 10) & (~is_wknd)).astype(int)
        X_out['Is_Evening_Peak'] = ((hours >= 17) & (hours <= 20) & (~is_wknd)).astype(int)
        X_out['Is_Peak_Hour'] = (X_out['Is_Morning_Peak'] | X_out['Is_Evening_Peak']).astype(int)
        
        return X_out

print(" TemporalFeatureExtractor compiled.")

 TemporalFeatureExtractor compiled.


### 2.2 `StationBaselineEncoder`
Learns station-level historical baseline volume during `.fit()` on the training set and maps them deterministically during `.transform()`. This eliminates training-serving skew for station profiles.

In [4]:
class StationBaselineEncoder(BaseEstimator, TransformerMixin):
    """
    Learns station baseline average traffic without target leakage.
    """
    def __init__(self, station_col='Station', default_avg=500.0):
        self.station_col = station_col
        self.default_avg = default_avg
        self.station_avg_map_ = {}

    def fit(self, X, y=None):
        # If y (target) is provided, compute mean per station
        if y is not None:
            temp_df = pd.DataFrame({'Station': X[self.station_col], 'Target': y})
            self.station_avg_map_ = temp_df.groupby('Station')['Target'].mean().to_dict()
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out['Station_AvgTraffic'] = X_out[self.station_col].map(
            lambda s: self.station_avg_map_.get(s, self.default_avg)
        )
        return X_out

print(" StationBaselineEncoder compiled.")

 StationBaselineEncoder compiled.


### 2.3 `LagAndRollingTransformer`
Extracts autoregressive temporal dependencies (`Lag_1h`, `Lag_2h`, `Lag_24h`) and computes moving average and volatility features (`Rolling_3h`, `Rolling_3h_Std`).

In [5]:
class LagAndRollingTransformer(BaseEstimator, TransformerMixin):
    """
    Generates multi-scale lag features and rolling window aggregates.
    """
    def __init__(self, target_col='Boarding_Count', group_col='Station'):
        self.target_col = target_col
        self.group_col = group_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.target_col in X_out.columns:
            # Batch / training mode
            X_out['Lag_1h'] = X_out.groupby(self.group_col)[self.target_col].shift(1)
            X_out['Lag_2h'] = X_out.groupby(self.group_col)[self.target_col].shift(2)
            X_out['Lag_24h'] = X_out.groupby(self.group_col)[self.target_col].shift(24)
            
            # Rolling statistics
            r3 = X_out.groupby(self.group_col)[self.target_col].rolling(window=3, min_periods=1)
            X_out['Rolling_3h'] = r3.mean().reset_index(level=0, drop=True)
            X_out['Rolling_3h_Std'] = r3.std().fillna(0).reset_index(level=0, drop=True)
            
            # Handle cold-start nulls with station baseline
            if 'Station_AvgTraffic' in X_out.columns:
                X_out['Lag_1h'].fillna(X_out['Station_AvgTraffic'] * 0.8, inplace=True)
                X_out['Lag_2h'].fillna(X_out['Station_AvgTraffic'] * 0.7, inplace=True)
                X_out['Lag_24h'].fillna(X_out['Station_AvgTraffic'], inplace=True)
                X_out['Rolling_3h'].fillna(X_out['Station_AvgTraffic'], inplace=True)
                X_out['Rolling_3h_Std'].fillna(0.0, inplace=True)
        return X_out

print(" LagAndRollingTransformer compiled.")

 LagAndRollingTransformer compiled.


### 2.4 `FeatureColumnSelector`
Ensures strict feature ordering and filters out raw string identifiers before passing data to the estimator.

In [6]:
FEATURE_ORDER = [
    'Hour', 'Hour_Sin', 'Hour_Cos', 'DayOfWeek', 'Day_Sin', 'Day_Cos', 'DayOfMonth',
    'WeekOfYear', 'Is_Weekend', 'Is_Morning_Peak', 'Is_Evening_Peak',
    'Is_Peak_Hour', 'Exit_Count', 'Lag_1h', 'Lag_2h', 'Lag_24h', 'Rolling_3h',
    'Rolling_3h_Std', 'Station_AvgTraffic'
]

class FeatureColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_columns=FEATURE_ORDER):
        self.feature_columns = feature_columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.feature_columns]

print(" FeatureColumnSelector compiled with 19 verified features.")

 FeatureColumnSelector compiled with 19 verified features.


---
## 3. End-to-End Pipeline Assembly

We assemble the full pipeline into a single Scikit-Learn `Pipeline` container.
Notice how cleanly the steps flow:

In [7]:
# Define the optimal hyperparameters from Notebook 04
xgb_regressor = xgb.XGBRegressor(
    n_estimators=350,
    learning_rate=0.04,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1
)

# Construct Master Production Pipeline
production_pipeline = Pipeline(steps=[
    ('temporal_extractor', TemporalFeatureExtractor(date_col='Date', hour_col='Hour')),
    ('station_encoder', StationBaselineEncoder(station_col='Station')),
    ('lag_transformer', LagAndRollingTransformer(target_col='Boarding_Count', group_col='Station')),
    ('feature_selector', FeatureColumnSelector(feature_columns=FEATURE_ORDER)),
    ('regressor', xgb_regressor)
])

print(" Complete End-to-End Machine Learning Pipeline Assembled:")
production_pipeline

 Complete End-to-End Machine Learning Pipeline Assembled:


,steps,"[('temporal_extractor', ...), ('station_encoder', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,date_col,'Date'
,hour_col,'Hour'
,station_col,'Station'
,default_avg,500.0
,target_col,'Boarding_Count'
,group_col,'Station'
,feature_columns,"['Hour', 'Hour_Sin', ...]"


---
## 4. Time-Series Split & Model Evaluation

In time-series forecasting, standard random cross-validation leaks future patterns into past training splits. We perform a strict chronological time-series split.

In [8]:
# Prepare input feature matrix and target vector
X = raw_df[['Date', 'Hour', 'Station', 'Exit_Count', 'Boarding_Count']].copy()
y = raw_df['Boarding_Count'].values

# 80% Train, 20% Holdout Test split based on chronological date
dates = sorted(X['Date'].unique())
split_idx = int(len(dates) * 0.8)
train_dates = dates[:split_idx]
test_dates = dates[split_idx:]

train_mask = X['Date'].isin(train_dates)
test_mask = X['Date'].isin(test_dates)

X_train, y_train = X[train_mask].copy(), y[train_mask]
X_test, y_test = X[test_mask].copy(), y[test_mask]

print(f"Training set size: {len(X_train)} rows ({train_dates[0]} to {train_dates[-1]})")
print(f"Testing set size:  {len(X_test)} rows ({test_dates[0]} to {test_dates[-1]})")

Training set size: 72360 rows (2025-08-01 to 2025-09-20)
Testing set size:  19920 rows (2025-09-21 to 2025-09-30)


In [9]:
print(" Fitting full production pipeline on training partition...")
production_pipeline.fit(X_train, y_train)
print(" Pipeline training complete!")

 Fitting full production pipeline on training partition...


C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_out['Lag_1h'].fillna(X_out['Station_AvgTraffic'] * 0.8, inplace=True)
C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

 Pipeline training complete!


### 4.1 Evaluate Pipeline Performance on Unseen Test Partition

In [10]:
# Predict on unseen test partition
y_pred = production_pipeline.predict(X_test)
y_pred_clean = np.maximum(0, y_pred)

# Metrics calculation
rmse = root_mean_squared_error(y_test, y_pred_clean)
mae = mean_absolute_error(y_test, y_pred_clean)
r2 = r2_score(y_test, y_pred_clean)
wmape = (np.sum(np.abs(y_test - y_pred_clean)) / np.sum(y_test)) * 100

print("===============================================")
print(" PRODUCTION PIPELINE EVALUATION ON TEST SET:")
print("===============================================")
print(f"  • R² Score (Variance Explained): {r2:.4f} ({r2*100:.2f}%)")
print(f"  • Root Mean Squared Error (RMSE): {rmse:.2f} passengers")
print(f"  • Mean Absolute Error (MAE):     {mae:.2f} passengers")
print(f"  • System-Wide WMAPE:             {wmape:.2f}%")
print("===============================================")

 PRODUCTION PIPELINE EVALUATION ON TEST SET:
  • R² Score (Variance Explained): 0.9955 (99.55%)
  • Root Mean Squared Error (RMSE): 32.01 passengers
  • Mean Absolute Error (MAE):     14.56 passengers
  • System-Wide WMAPE:             4.08%


C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_out['Lag_1h'].fillna(X_out['Station_AvgTraffic'] * 0.8, inplace=True)
C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

---
## 5. Serializing & Exporting the Production Pipeline Artifact

Instead of exporting only model weights and leaving feature extraction to external scripts, we serialize the **entire pipeline bundle** along with version metadata.

In [11]:
pipeline_export_path = '../models/production_pipeline.joblib'
metadata_export_path = '../models/pipeline_metadata.json'

# 1. Save serialized pipeline
joblib.dump(production_pipeline, pipeline_export_path)

# 2. Save accompanying metadata
metadata = {
    "pipeline_name": "namma_metro_passenger_flow_pipeline",
    "version": "2.1.0",
    "trained_at": datetime.now().isoformat(),
    "model_type": "XGBRegressor",
    "feature_count": len(FEATURE_ORDER),
    "features": FEATURE_ORDER,
    "evaluation_metrics": {
        "r2_score": round(float(r2), 4),
        "rmse": round(float(rmse), 2),
        "mae": round(float(mae), 2),
        "wmape_pct": round(float(wmape), 2)
    }
}

import json
with open(metadata_export_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f" Production pipeline exported to: {pipeline_export_path}")
print(f" Pipeline metadata saved to:     {metadata_export_path}")

 Production pipeline exported to: ../models/production_pipeline.joblib
 Pipeline metadata saved to:     ../models/pipeline_metadata.json


---
## 6. Real-Time Inference Simulation

To prove production readiness, we load the saved `.joblib` pipeline and make a prediction on a raw, un-transformed dictionary as sent by a user from the web UI.

In [12]:
# 1. Load pipeline from disk
loaded_pipeline = joblib.load(pipeline_export_path)

# 2. Simulate raw user query from FastAPI web frontend
live_request_sample = pd.DataFrame([{
    'Station': 'Nadaprabhu Kempegowda Station, Majestic',
    'Date': '2025-09-25',
    'Hour': 9,         # 9:00 AM Morning Rush
    'Exit_Count': 420,
    'Boarding_Count': 2200  # Prior hour proxy
}])

# 3. Run single-step inference directly through pipeline
predicted_volume = loaded_pipeline.predict(live_request_sample)[0]
print("===========================================================")
print(" INFERENCE SIMULATION RESULT:")
print(f"  • Station:   {live_request_sample['Station'].values[0]}")
print(f"  • Timestamp: {live_request_sample['Date'].values[0]} at {live_request_sample['Hour'].values[0]}:00 AM")
print(f"  • Predicted Boarding Flow: {int(round(predicted_volume)):,} passengers / hour")
print("===========================================================")

 INFERENCE SIMULATION RESULT:
  • Station:   Nadaprabhu Kempegowda Station, Majestic
  • Timestamp: 2025-09-25 at 9:00 AM
  • Predicted Boarding Flow: 2,377 passengers / hour


C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_out['Lag_1h'].fillna(X_out['Station_AvgTraffic'] * 0.8, inplace=True)
C:\Users\tejas\AppData\Local\Temp\ipykernel_4540\4278069000.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

---
##  Summary & Takeaways for ML Engineers

1. **Zero Training-Serving Skew:** All calendar transformations, cyclical trigonometry, station baselines, and lags are bundled in the exact same estimator graph for training and serving.
2. **Production-Ready Artifact:** The deployment API only needs to load `production_pipeline.joblib` — no duplicated feature calculation scripts required.
3. **Extensibility:** New features (e.g. weather data, holiday calendar, transit delays) can be added as isolated transformers into the `Pipeline` without altering downstream regression logic.